# Stock Market Trend Analysis (SMA & EMA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Phuvanenthran-P/data-analytics-projects/blob/main/stock-market-analysis/stock_market_analysis.ipynb)

This notebook fetches intraday stock data from Yahoo Finance and analyzes price trends using a 50-period Simple Moving Average (SMA) and Exponential Moving Average (EMA). It includes static and interactive charts, crossover detection, and a timed live refresh.

**Tools:** Python, Pandas, yfinance, Matplotlib, Plotly

**How to use in Colab:** click **File → Save a copy in Drive**, then run **Runtime → Run all**.

## 1. Install and import libraries
The latest `yfinance` is installed because older versions often fail with Yahoo Finance errors.

In [ ]:
!pip install -q -U yfinance plotly

In [ ]:
import time
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import clear_output

## 2. Settings
Change `TICKER` to analyze any listed stock (e.g. `MSFT`, `GOOGL`, `TSLA`).

In [ ]:
TICKER = "AAPL"
WINDOW = 50  # moving average period

## 3. Helper functions
- `fetch_data` downloads 1-minute data, retries if Yahoo is busy, and falls back to the last 5 days when the market is closed (weekends and holidays).
- `add_indicators` cleans the data and adds SMA and EMA columns.

In [ ]:
def fetch_data(ticker, retries=3):
    for attempt in range(1, retries + 1):
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period="1d", interval="1m")
            if data.empty:
                # Market closed today: use the last 5 days instead
                data = stock.history(period="5d", interval="1m")
            data = data.dropna()
            if not data.empty:
                return data
            print(f"Attempt {attempt}: no data returned for {ticker}.")
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
        time.sleep(5 * attempt)
    raise RuntimeError(
        f"Could not fetch data for {ticker}. Check the ticker symbol, "
        "wait a few minutes if Yahoo is rate-limiting, then re-run."
    )


def add_indicators(data, window=WINDOW):
    data = data.dropna().copy()
    data["SMA"] = data["Close"].rolling(window=window).mean()
    data["EMA"] = data["Close"].ewm(span=window, adjust=False).mean()
    return data

## 4. Collect and prepare data

In [ ]:
raw = fetch_data(TICKER)
print(f"Rows downloaded: {len(raw)}")
print("Missing values per column:")
print(raw.isnull().sum())

df = add_indicators(raw)
df[["Close", "SMA", "EMA"]].tail()

## 5. Crossover signals
When the EMA moves above the SMA it suggests upward momentum (bullish); below suggests downward momentum (bearish).

In [ ]:
valid = df.dropna(subset=["SMA"]).copy()
valid["Signal"] = (valid["EMA"] > valid["SMA"]).astype(int)
valid["Crossover"] = valid["Signal"].diff()

bullish = valid[valid["Crossover"] == 1]
bearish = valid[valid["Crossover"] == -1]

print(f"Bullish crossovers (EMA crosses above SMA): {len(bullish)}")
print(f"Bearish crossovers (EMA crosses below SMA): {len(bearish)}")

## 6. Static chart (Matplotlib)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df["Close"], label="Stock Price", color="blue")
plt.plot(df["SMA"], label=f"{WINDOW}-Period SMA", color="orange")
plt.plot(df["EMA"], label=f"{WINDOW}-Period EMA", color="red")
plt.title(f"{TICKER} Price with SMA and EMA")
plt.xlabel("Time")
plt.ylabel("Price (USD)")
plt.legend(loc="upper left")
plt.show()

## 7. Interactive chart (Plotly)
Hover over points, zoom, and pan to explore.

In [ ]:
def build_figure(data):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=data.index, y=data["Close"], mode="lines", name="Stock Price"))
    fig.add_trace(go.Scatter(x=data.index, y=data["SMA"], mode="lines", name=f"{WINDOW}-Period SMA"))
    fig.add_trace(go.Scatter(x=data.index, y=data["EMA"], mode="lines", name=f"{WINDOW}-Period EMA"))
    fig.update_layout(
        title=f"{TICKER} Price with SMA & EMA",
        xaxis_title="Time",
        yaxis_title="Price (USD)",
        template="plotly_dark",
    )
    return fig

build_figure(df).show()

## 8. Live refresh (limited)
Re-fetches the data every minute for a fixed number of updates, then stops. Prices only change while the market is open (US market: 7:00 PM to 1:30 AM IST). Increase `REFRESHES` for longer monitoring.

In [ ]:
REFRESHES = 3
SECONDS_BETWEEN = 60

for i in range(REFRESHES):
    live = add_indicators(fetch_data(TICKER))

    clear_output(wait=True)
    print(f"Update {i + 1} of {REFRESHES}")
    build_figure(live).show()

    if i < REFRESHES - 1:
        time.sleep(SECONDS_BETWEEN)

## Conclusion
The notebook shows how moving averages smooth intraday price noise and highlight trend direction. The EMA reacts faster than the SMA, and crossovers between them give simple momentum signals.

**Limitations:** Yahoo Finance data can be delayed and rate-limited, and moving averages lag price, so they are not predictive. This is an educational project, not financial advice.

**Next steps:** compare multiple stocks, add RSI and MACD, and deploy as a Streamlit app.